In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import gammaln, logsumexp

from zne import fold_circuit
from circuits import generate_ising_benchmark

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.noise import NoiseModel, depolarizing_error

In [ ]:
def negative_log_likelihood(params, y_data, lambda_scales, sigma_meas, k_max=15):
    expval_ideal, lambda_base, mu, sigma = params
    
    if lambda_base <= 0 or sigma <= 0 or abs(expval_ideal) > 1.2:
        return np.inf
    
    nll_total = 0.0
    k_arr = np.arange(k_max + 1)
    
    for y, lambda_scale in zip(y_data, lambda_scales):
        lambda_rate = lambda_base * lambda_scale
        log_poisson = k_arr * np.log(lambda_rate + 1e-12) - lambda_rate - gammaln(k_arr + 1)
        
        variance = sigma_meas**2 + k_arr * (sigma**2)
        mean_shift = expval_ideal - k_arr * mu
        
        log_gaussian = -0.5 * np.log(2 * np.pi * variance) - ((y - mean_shift)**2) / (2 * variance)
        
        log_terms = log_poisson + log_gaussian
        log_p_y = logsumexp(log_terms)
        
        nll_total -= log_p_y
        
    return nll_total

def fit_zne_mle(measurements_dict, shots):
    y_data = []
    lambda_scales = []
    for lambda_scale, values in measurements_dict.items():
        y_data.extend(values)
        lambda_scales.extend([lambda_scale] * len(values))
        
    y_data = np.array(y_data)
    lambda_scales = np.array(lambda_scales)
    
    sigma_meas = 1.0 / np.sqrt(shots)
    
    guess_expval_ideal = np.mean(measurements_dict[1]) + 0.1 
    initial_guess = [guess_expval_ideal, 1.0, 0.1, 0.05]
    
    print("Starting MLE Optimization...")
    result = minimize(
        negative_log_likelihood,
        initial_guess,
        args=(y_data, lambda_scales, sigma_meas),
        method="Nelder-Mead", 
        options={"maxiter": 1000, "xatol": 1e-5, "fatol": 1e-5}
    )
    
    if result.success:
        expval_ideal, gamma, mu, sigma = result.x
        
        return {
            "expval_ideal": expval_ideal,
            "gamma": gamma,
            "mu": mu,
            "sigma": sigma,
            "nll": result.fun
        }
    else:
        raise ValueError("Optimizer failed to converge.")

In [ ]:
noise_model = NoiseModel()
error_1q = depolarizing_error(1E-2, 1)
noise_model.add_all_qubit_quantum_error(error_1q, ["x", "y", "z", "h", "s", "t"])
error_2q = depolarizing_error(1E-1, 2)
noise_model.add_all_qubit_quantum_error(error_2q, ["cx"])

n_qubits = 4
qc, obs = generate_ising_benchmark(n_qubits=n_qubits, trotter_steps=2)

lambda_scales = [2*k+1 for k in range(10)]
batches = 1
shots_per_batch = 1000
shots = shots_per_batch * batches

target = "probability"

if target == "observable":
    estimator = AerEstimator(
        backend_options={"noise_model": noise_model},
        run_options={"shots": shots_per_batch}
    )
elif target == "probability":
    aer_simulator = AerSimulator(noise_model=noise_model)

print("--- Step 1: Gathering Data ---")
data = {}

for lambda_scale in lambda_scales:
    qc_scaled = fold_circuit(qc, lambda_scale)

    if target == "observable":
        result = estimator.run([qc_scaled] * batches, [observable] * batches, optimization_level=0).result()
        values = result.values
        data[lambda_scale] = values

        print(f"Scale λ={lambda_scale}: Mean E = {np.mean(values):.4f}, Variance = {np.var(values):.4f}")
    elif target == "probability":
        qc_scaled.measure_all()
        
        result = aer_simulator.run(qc_scaled, shots=shots, optimization_level=0).result()
        counts = result.get_counts()
        p0 = counts["0"*n_qubits]/shots
        data[lambda_scale] = [p0]
    
        print(f"Scale λ={lambda_scale}: Mean E = {p0:.4f}, Variance = {p0:.4f}")

print("\n--- Step 2: Optimizing MLE Model ---")

results = fit_zne_mle(data, shots=shots_per_batch)

print("\n--- Fit Results ---")
print(f"Extrapolated E(ideal): {results["expval_ideal"]:.4f}")
print(f"Base Error Rate (γ):   {results["gamma"]:.4f} errors/circuit at λ=1")
print(f"Avg Error Impact (μ):  {results["mu"]:.4f} signal degradation per error")
print(f"Error Impact Var (σ):  {results["sigma"]:.4f}")
print(f"Final NLL Score:       {results["nll"]:.4f}")